In [1]:
!pip install mne


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 29.6 MB/s eta 0:00:00


In [35]:
import mne
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

edf_path = "/content/drive/MyDrive/files/S071/S071R12.edf"

raw = mne.io.read_raw_edf(edf_path, preload=True)

sfreq = int(raw.info["sfreq"])

print("Sampling rate:", sfreq)

Extracting EDF parameters from /content/drive/MyDrive/files/S071/S071R12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Sampling rate: 160


In [36]:
MOTOR_CHS = [
'Fc1.','Fc3.','Fcz.','Fc2.','Fc4.',
'C1..','C3..','Cz..','C2..','C4..',
'Cp1.','Cp3.','Cpz.','Cp2.','Cp4.'
]

used = [c for c in MOTOR_CHS if c in raw.ch_names]

picks = mne.pick_channels(raw.info['ch_names'], include=used)

data = raw.get_data(picks=picks)

print("Channels used:", used)
print("Data shape:", data.shape)

Channels used: ['Fc1.', 'Fc3.', 'Fcz.', 'Fc2.', 'Fc4.', 'C1..', 'C3..', 'Cz..', 'C2..', 'C4..', 'Cp1.', 'Cp3.', 'Cpz.', 'Cp2.', 'Cp4.']
Data shape: (15, 20000)


In [37]:
event_dict = {
    "T0":0,
    "T1":1,
    "T2":2
}

events, event_id = mne.events_from_annotations(raw)

print("Event mapping:", event_id)
print("First 10 events:\n", events[:10])

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Event mapping: {np.str_('T0'): 1, np.str_('T1'): 2, np.str_('T2'): 3}
First 10 events:
 [[   0    0    1]
 [ 672    0    3]
 [1328    0    1]
 [2000    0    2]
 [2656    0    1]
 [3328    0    3]
 [3984    0    1]
 [4656    0    2]
 [5312    0    1]
 [5984    0    3]]


In [38]:
import torch
import torch.nn as nn
class EEGNet(nn.Module):
    def __init__(self, ch, t, F1=16, D=2, dropout=0.5):
        super(EEGNet, self).__init__()

        F2 = F1 * D   # 32

        # Block 1: Temporal conv + Depthwise spatial conv
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F2, (ch, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout)
        )

        # Block 2: Separable conv (depthwise + pointwise)
        self.block2 = nn.Sequential(
            nn.Conv2d(F2, F2, (1, 16), padding=(0, 8), groups=F2, bias=False),
            nn.Conv2d(F2, F2, (1, 1),  bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout)
        )

        # Block 3: Extra separable conv (no pooling)
        self.block3 = nn.Sequential(
            nn.Conv2d(F2, F2, (1, 8), padding=(0, 4), groups=F2, bias=False),
            nn.Conv2d(F2, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.Dropout(dropout)
        )

        # Dynamically compute FC input size via dummy forward pass
        # This avoids any shape mismatch regardless of ch or t
        with torch.no_grad():
            dummy  = torch.zeros(1, 1, ch, t)
            dummy  = self.block1(dummy)
            dummy  = self.block2(dummy)
            dummy  = self.block3(dummy)
            fc_in  = dummy.flatten(start_dim=1).shape[1]

        print(f"  EEGNet FC input size: {fc_in}")
        self.fc = nn.Linear(fc_in, 3)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.fc(x.flatten(start_dim=1))


In [39]:
n_ch = 15
n_t  = 353   # <-- was 161, must be 353 to get fc_in=384

eegnet_model = EEGNet(n_ch, n_t).to(device)

eegnet_model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/EEGNET~ 66/eegnet_best.pt",
        map_location=device
    )
)


  EEGNet FC input size: 384


<All keys matched successfully>

In [40]:
eegnet_model.eval()
print("EEGNet loaded successfully")


EEGNet loaded successfully


In [41]:
CLASS_NAMES = ["REST","LEFT","RIGHT"]

event_map = {
    event_id["T0"]:0,
    event_id["T1"]:1,
    event_id["T2"]:2
}

In [42]:
import torch
import numpy as np

# ================================================================
# CONFIG — must match training exactly
# ================================================================

tmin  = -0.2
tmax  =  2.0
sfreq = 160
n_ch  = 15
n_t   = int((tmax - tmin) * sfreq) + 1  # = 353

correct = 0
total   = 0
preds   = []
truths  = []

for ev in events:

    event_sample = ev[0]
    label        = ev[2]
    true_class   = event_map.get(label, None)

    if true_class is None:
        continue

    start   = int(event_sample + tmin * sfreq)
    end     = start + n_t                          # 353 samples

    segment = data[:, start:end]

    if segment.shape[1] != n_t:
        continue

    # Normalize same as training
    segment = segment.astype(np.float32)
    segment = (segment - segment.mean()) / (segment.std() + 1e-8)

    xb = torch.tensor(
        segment[np.newaxis, np.newaxis],           # (1, 1, ch, t)
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():
        probs = torch.softmax(eegnet_model(xb), dim=1).cpu().numpy().squeeze()

    pred = np.argmax(probs)

    preds.append(pred)
    truths.append(true_class)

    if pred == true_class:
        correct += 1
    total += 1

    print(f"True: {CLASS_NAMES[true_class]:<6} "
          f"| Pred: {CLASS_NAMES[pred]:<6} "
          f"| REST={probs[0]:.2f} LEFT={probs[1]:.2f} RIGHT={probs[2]:.2f}")

print(f"\nTotal  : {total}")
print(f"Correct: {correct}")
print(f"Accuracy: {correct/total:.4f}")

True: RIGHT  | Pred: RIGHT  | REST=0.18 LEFT=0.34 RIGHT=0.48
True: REST   | Pred: REST   | REST=0.45 LEFT=0.43 RIGHT=0.12
True: LEFT   | Pred: RIGHT  | REST=0.08 LEFT=0.39 RIGHT=0.53
True: REST   | Pred: REST   | REST=0.60 LEFT=0.17 RIGHT=0.23
True: RIGHT  | Pred: RIGHT  | REST=0.06 LEFT=0.22 RIGHT=0.73
True: REST   | Pred: REST   | REST=0.55 LEFT=0.18 RIGHT=0.26
True: LEFT   | Pred: LEFT   | REST=0.15 LEFT=0.75 RIGHT=0.10
True: REST   | Pred: REST   | REST=0.63 LEFT=0.05 RIGHT=0.32
True: RIGHT  | Pred: RIGHT  | REST=0.12 LEFT=0.39 RIGHT=0.50
True: REST   | Pred: REST   | REST=0.91 LEFT=0.02 RIGHT=0.07
True: LEFT   | Pred: REST   | REST=0.85 LEFT=0.13 RIGHT=0.01
True: REST   | Pred: REST   | REST=0.65 LEFT=0.09 RIGHT=0.26
True: LEFT   | Pred: LEFT   | REST=0.14 LEFT=0.69 RIGHT=0.18
True: REST   | Pred: REST   | REST=0.84 LEFT=0.10 RIGHT=0.07
True: RIGHT  | Pred: RIGHT  | REST=0.33 LEFT=0.07 RIGHT=0.60
True: REST   | Pred: RIGHT  | REST=0.16 LEFT=0.36 RIGHT=0.47
True: RIGHT  | Pred: RIG